In [3]:
!pip install xgboost[dask] --upgrade




[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import xgboost.dask as dxgb

c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\cupy\_environment.py:217: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(
c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
!nvidia-smi

Sat Apr  5 15:03:31 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 555.99                 Driver Version: 555.99         CUDA Version: 12.5     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   44C    P3             19W /   40W |       0MiB /   6144MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [10]:
import pynvml

pynvml.nvmlDeviceGetUtilizationRates().gpu

TypeError: nvmlDeviceGetUtilizationRates() missing 1 required positional argument: 'handle'

In [ ]:
import xgboost as xgb
import dask
from dask.distributed import Client
import pynvml
import warnings

def check_gpu_cpu_setup():
    print("========== Environment Check ==========")
    
    # Step 1: Check Dask version
    print(f"Dask version: {dask.__version__}")
    
    # Step 2: Start Dask client safely
    try:
        client = Client(diagnostics_port=None)  # avoid GPU monitoring errors
        print("Dask Client started successfully ✅")
    except Exception as e:
        print(f"Failed to start Dask Client ❌: {e}")
        return
    
    # Step 3: Check if xgboost.dask exists
    if hasattr(xgb, "dask"):
        print("xgboost.dask is available ✅")
    else:
        print("xgboost.dask is NOT available ❌ (Update your xgboost library)")
        return
    
    # Step 4: Check GPU availability
    try:
        # pynvml.nvmlInit()
        # device_count = pynvml.nvmlDeviceGetCount()
        # print(f"Number of GPUs detected: {device_count} ✅")
        # for i in range(device_count):
        #     handle = pynvml.nvmlDeviceGetHandleByIndex(i)
        #     gpu_name = pynvml.nvmlDeviceGetName(handle)
        #     mem_info = pynvml.nvmlDeviceGetMemoryInfo(handle)
        #     print(f"GPU {i}: {gpu_name.decode('utf-8')} - Total Memory: {mem_info.total / 1024**3:.2f} GB")
    except pynvml.NVMLError as nvml_error:
        print(f"GPU check failed ❌: {nvml_error}")
    except Exception as e:
        print(f"Unexpected error during GPU check: {e}")

    # Step 5: Final statement
    print("========================================")

# Run the function
if __name__ == "__main__":
    # Optional: suppress unnecessary warnings
    warnings.filterwarnings("ignore", category=RuntimeWarning)
    
    check_gpu_cpu_setup()


========== Environment Check ==========
Dask version: 2025.3.0


2025-04-05 15:04:47,404 - tornado.application - ERROR - Exception in callback <bound method SystemMonitor.update of <SystemMonitor: cpu: 0 memory: 221 MB fds: N/A>>
Traceback (most recent call last):
  File "c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\tornado\ioloop.py", line 919, in _run
    val = self.callback()
  File "c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\distributed\system_monitor.py", line 210, in update
    gpu_metrics = nvml.real_time()
  File "c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\distributed\diagnostics\nvml.py", line 370, in real_time
    "utilization": _get_utilization(h),
  File "c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\distributed\diagnostics\nvml.py", line 339, in _get_utilization
    return pynvml.nvmlDeviceGetUtilizationRates(h).gpu
  File "c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\pynvml\nvml.py", line 2137, 

Dask Client started successfully ✅
xgboost.dask is NOT available ❌ (Update your xgboost library)


2025-04-05 15:04:49,900 - tornado.application - ERROR - Exception in callback <bound method SystemMonitor.update of <SystemMonitor: cpu: 0 memory: 223 MB fds: N/A>>
Traceback (most recent call last):
  File "c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\tornado\ioloop.py", line 919, in _run
    val = self.callback()
  File "c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\distributed\system_monitor.py", line 210, in update
    gpu_metrics = nvml.real_time()
  File "c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\distributed\diagnostics\nvml.py", line 370, in real_time
    "utilization": _get_utilization(h),
  File "c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\distributed\diagnostics\nvml.py", line 339, in _get_utilization
    return pynvml.nvmlDeviceGetUtilizationRates(h).gpu
  File "c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\pynvml\nvml.py", line 2137, 

In [ ]:
import os
os.environ["DASK_DISTRIBUTED__DIAGNOSTICS__NVML"] = "False"

from dask.distributed import Client
import xgboost.dask as xgb
import dask.array as da

# Start Dask
client = Client()

# Dummy data
X = da.random.random((1000, 20), chunks=(100, 20))
y = da.random.randint(0, 2, size=(1000,), chunks=(100,))

# Train with Dask XGBoost
dtrain = xgb.DaskDMatrix(client, X, y)
output = xgb.train(client, {}, dtrain, num_boost_round=10)

print("Training completed 🚀")


c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\cupy\_environment.py:217: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(
2025-04-05 15:07:09,906 - tornado.application - ERROR - Exception in callback <bound method SystemMonitor.update of <SystemMonitor: cpu: 0 memory: 223 MB fds: N/A>>
Traceback (most recent call last):
  File "c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\tornado\ioloop.py", line 919, in _run
    val = self.callback()
  File "c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\distributed\system_monitor.py", line 210, in update
    gpu_metrics = nvml.real_time()
  File "c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\distributed\diagnostics\nvml.py", line 370, in real_time
    "utilization": _get_utilization(h),
  File "c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\distribu

RuntimeError: Cluster failed to start: Unknown Error

2025-04-05 15:07:15,209 - tornado.application - ERROR - Exception in callback <bound method SystemMonitor.update of <SystemMonitor: cpu: 0 memory: 222 MB fds: N/A>>
Traceback (most recent call last):
  File "c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\tornado\ioloop.py", line 919, in _run
    val = self.callback()
  File "c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\distributed\system_monitor.py", line 210, in update
    gpu_metrics = nvml.real_time()
  File "c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\distributed\diagnostics\nvml.py", line 370, in real_time
    "utilization": _get_utilization(h),
  File "c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\distributed\diagnostics\nvml.py", line 339, in _get_utilization
    return pynvml.nvmlDeviceGetUtilizationRates(h).gpu
  File "c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\pynvml\nvml.py", line 2137, 

2025-04-05 15:07:15,246 - tornado.application - ERROR - Exception in callback <bound method SystemMonitor.update of <SystemMonitor: cpu: 0 memory: 222 MB fds: N/A>>
Traceback (most recent call last):
  File "c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\tornado\ioloop.py", line 919, in _run
    val = self.callback()
  File "c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\distributed\system_monitor.py", line 210, in update
    gpu_metrics = nvml.real_time()
  File "c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\distributed\diagnostics\nvml.py", line 370, in real_time
    "utilization": _get_utilization(h),
  File "c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\distributed\diagnostics\nvml.py", line 339, in _get_utilization
    return pynvml.nvmlDeviceGetUtilizationRates(h).gpu
  File "c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\pynvml\nvml.py", line 2137, 

In [ ]:
import os
import logging

# Mute the NVML GPU errors
logging.getLogger("distributed.diagnostics.nvml").setLevel(logging.CRITICAL)

# Disable Dask GPU diagnostics
os.environ["DASK_DISTRIBUTED__DIAGNOSTICS__NVML"] = "False"

from dask.distributed import Client
import xgboost.dask as xgb
import dask.array as da

# Start Dask client
client = Client()
print("Dask client started:", client)

# Dummy data
X_train = da.random.random(size=(10000, 20), chunks=(1000, 20))
y_train = da.random.randint(0, 2, size=(10000,), chunks=(1000,))

# DaskDMatrix
dtrain = xgb.DaskDMatrix(client, X_train, y_train)

# Parameters
params = {
    'tree_method': 'hist',
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'verbosity': 1
}

# Training
output = xgb.train(
    client,
    params,
    dtrain,
    num_boost_round=50,
    evals=[(dtrain, 'train')]
)

print("Training finished 🚀")


c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\distributed\node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 56436 instead
  warnings.warn(


Dask client started: <Client: 'tcp://127.0.0.1:56439' processes=4 threads=16, memory=13.86 GiB>
Training finished 🚀


2025-04-05 14:44:35,889 - tornado.application - ERROR - Exception in callback functools.partial(<function TCPServer._handle_connection.<locals>.<lambda> at 0x0000028A4979BF40>, <Task finished name='Task-55859' coro=<BaseTCPListener._handle_stream() done, defined at c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\distributed\comm\tcp.py:655> exception=MemoryError((6073139484287059271,), dtype('uint8'))>)
Traceback (most recent call last):
  File "c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\tornado\ioloop.py", line 738, in _run_callback
    ret = callback()
  File "c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\tornado\tcpserver.py", line 387, in <lambda>
    gen.convert_yielded(future), lambda f: f.result()
  File "c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\distributed\comm\tcp.py", line 667, in _handle_stream
    await self.on_connection(comm)
  File "c:\Users\saila\AppData\Loc

In [3]:
!pip install bokeh>=3.1.0

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!nvcc -V

'nvcc' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
import tensorflow as tf
from tensorflow.python.platform import build_info as build
print(f"tensorflow version: {tf.__version__}")
print(f"Cuda Version: {build.build_info['cuda_version']}")
print(f"Cudnn version: {build.build_info['cudnn_version']}")

ModuleNotFoundError: No module named 'tensorflow'

In [4]:
import torch

print(torch.version.cuda)

12.4


In [5]:
print(torch.cuda.is_available())  # True if GPU is available and usable
print(torch.cuda.get_device_name(0))  # Name of your GPU

True
NVIDIA GeForce RTX 3060 Laptop GPU


In [2]:
!pip install xgboost_ray


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import pandas as pd

pd.read_parquet("./data/processed/final_sales_data_val").head()

,id,item_id,dept_id,cat_id,store_id,state_id,day,sales,lag_1,lag_7,lag_28,rolling_mean_7,rolling_mean_14
__null_dask_index__,,,,,,,,,,,,,
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,1,0.0,NaN,NaN,NaN,NaN,NaN
1,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,2,0.0,0.0,NaN,NaN,NaN,NaN
2,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,3,0.0,0.0,NaN,NaN,NaN,NaN
3,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,4,0.0,0.0,NaN,NaN,NaN,NaN
4,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,5,0.0,0.0,NaN,NaN,NaN,NaN


In [8]:
import lightgbm
print(lightgbm.__version__)

c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\cupy\_environment.py:217: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(
c:\Users\saila\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


4.6.0
